In [ ]:
# =======================================================
# Lab 3: Uninformed Search Implementations
# =======================================================
from collections import deque
import heapq

# Section 3.2 Topology
graph = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F"],
    "D": [],
    "E": ["G"],
    "F": [],
    "G": []
}

# --- Task 1: BFS and DFS ---
def bfs(start, goal):
    q = deque([(start, [start])])
    visited = {start}
    expansion_order = []
    while q:
        node, path = q.popleft()
        expansion_order.append(node)
        if node == goal:
            return path, expansion_order
        for n in graph[node]:
            if n not in visited:
                visited.add(n)
                q.append((n, path + [n]))
    return None, expansion_order

def dfs(start, goal):
    stack = [(start, [start])]
    visited = set()
    expansion_order = []
    while stack:
        node, path = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        expansion_order.append(node)
        if node == goal:
            return path, expansion_order
        for n in reversed(graph[node]):
            if n not in visited:
                stack.append((n, path + [n]))
    return None, expansion_order

print("--- Lab 3 Task 1: BFS & DFS Comparison ---")
bfs_path, bfs_order = bfs("A", "G")
dfs_path, dfs_order = dfs("A", "G")
print(f"BFS Path: {bfs_path} | Expansion Order: {bfs_order}")
print(f"DFS Path: {dfs_path} | Expansion Order: {dfs_order}")

# --- Task 2: DLS and IDDFS ---
def dls(node, goal, limit, path):
    if node == goal:
        return "FOUND", path
    if limit == 0:
        return "CUTOFF", None

    cutoff_occurred = False
    for nxt in graph[node]:
        if nxt not in path:  # Path-local cycle prevention
            res, sol = dls(nxt, goal, limit - 1, path + [nxt])
            if res == "FOUND":
                return "FOUND", sol
            if res == "CUTOFF":
                cutoff_occurred = True

    return ("CUTOFF" if cutoff_occurred else "FAILURE"), None

def iddfs(start, goal, max_depth):
    history = []
    for limit in range(max_depth + 1):
        status, sol = dls(start, goal, limit, [start])
        history.append((limit, status, sol))
        if status == "FOUND":
            return history, sol
    return history, None

print("\n--- Lab 3 Task 2: DLS & IDDFS (Limits 1 to 4) ---")
for lim in [1, 2, 3, 4]:
    status, sol = dls("A", "G", lim, ["A"])
    print(f"DLS at Limit {lim}: Status = {status:<8} | Solution = {sol}")

iddfs_history, iddfs_sol = iddfs("A", "G", 4)
print(f"\nIDDFS Search Progression up to Depth 4 (Final Path: {iddfs_sol}):")
for lim, stat, sol in iddfs_history:
    print(f"  Iteration Depth {lim}: {stat} -> {sol}")

# --- Task 3: Weighted Graph and UCS vs BFS ---
# Section 3.7 Corridor Extension: corridor C -> G added with cost 9
weighted_graph = {
    "A": [("B", 1), ("C", 3)],
    "B": [("D", 5), ("E", 1)],
    "C": [("F", 4), ("G", 9)],
    "D": [],
    "E": [("G", 1)],
    "F": [],
    "G": []
}

def ucs(start, goal):
    pq = [(0, start, [start])]
    best = {start: 0}
    expansion_order = []

    while pq:
        cost, node, path = heapq.heappop(pq)
        expansion_order.append(node)
        if node == goal:
            return cost, path, expansion_order
        if cost > best[node]:
            continue
        for nxt, weight in weighted_graph[node]:
            new_cost = cost + weight
            if new_cost < best.get(nxt, float('inf')):
                best[nxt] = new_cost
                heapq.heappush(pq, (new_cost, nxt, path + [nxt]))
    return float('inf'), None, expansion_order

ucs_cost, ucs_path, ucs_order = ucs("A", "G")
print("\n--- Lab 3 Task 3: UCS Minimum Cost Corridor ---")
print(f"UCS Optimal Cost : {ucs_cost}")
print(f"UCS Optimal Path : {ucs_path}")
print(f"Processing Order : {ucs_order}")

--- Lab 3 Task 1: BFS & DFS Comparison ---
BFS Path: ['A', 'B', 'E', 'G'] | Expansion Order: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
DFS Path: ['A', 'B', 'E', 'G'] | Expansion Order: ['A', 'B', 'D', 'E', 'G']

--- Lab 3 Task 2: DLS & IDDFS (Limits 1 to 4) ---
DLS at Limit 1: Status = CUTOFF   | Solution = None
DLS at Limit 2: Status = CUTOFF   | Solution = None
DLS at Limit 3: Status = FOUND    | Solution = ['A', 'B', 'E', 'G']
DLS at Limit 4: Status = FOUND    | Solution = ['A', 'B', 'E', 'G']

IDDFS Search Progression up to Depth 4 (Final Path: ['A', 'B', 'E', 'G']):
  Iteration Depth 0: CUTOFF -> None
  Iteration Depth 1: CUTOFF -> None
  Iteration Depth 2: CUTOFF -> None
  Iteration Depth 3: FOUND -> ['A', 'B', 'E', 'G']

--- Lab 3 Task 3: UCS Minimum Cost Corridor ---
UCS Optimal Cost : 3
UCS Optimal Path : ['A', 'B', 'E', 'G']
Processing Order : ['A', 'B', 'E', 'C', 'G']
